[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Red1-Rahman/NiriZan/blob/main/experiments/02_rag_triad_metrics.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https://github.com/Red1-Rahman/NiriZan/blob/main/experiments/02_rag_triad_metrics.ipynb)

# Experiment 02: RAG Triad Metrics
**Phase 2 Exploration**: Validating the `Metric` plugin interface, the RAG Triad (context relevance, groundedness, answer relevance), and the Metric Dispatcher against real traces from `examples/rag_pipeline_demo`.

Two scoring backends are explored:
1. **Embedding similarity** (free, local, no API key) — the default, rigorous baseline.
2. **Gemini LLM-as-judge** (free tier, optional) — closer to what RAGAS/ARES do in practice, used here to cross-check the embedding scores rather than replace them.

## 1. Environment Setup
Install `nirizan` from `main`, plus `sentence-transformers` for embedding-based scoring (free, runs locally/on Colab's GPU) and `google-generativeai` for the optional Gemini judge cross-check.

In [1]:
!pip install -q pydantic>=2.7 "git+https://github.com/Red1-Rahman/NiriZan.git@main#egg=nirizan"
!pip install -q sentence-transformers google-generativeai

from __future__ import annotations

import asyncio
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Protocol
from uuid import UUID, uuid4

from pydantic import BaseModel, ConfigDict, Field
import numpy as np

from nirizan.instrumentation.sdk import init_tracer, trace_span
from nirizan.instrumentation.spans import Span, SpanKind, Trace
from nirizan.orchestrator.collector import CollectorExporter, TraceCollector
from nirizan.storage.trace_repository import SQLiteTraceRepository

print("NiriZan Phase 1 components loaded successfully!")

NiriZan Phase 1 components loaded successfully!


## 2. Load the Embedding Model
`all-mpnet-base-v2`: best-quality sentence-transformers model that comfortably fits a free-tier T4 GPU for single-pair scoring (768-dim embeddings, strong STS-benchmark performance). This is the backbone of the reference-free RAG Triad scorer below.

In [2]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

embedder = SentenceTransformer("all-mpnet-base-v2", device=device)
print(f"Loaded all-mpnet-base-v2 ({embedder.get_sentence_embedding_dimension()}-dim embeddings)")

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded all-mpnet-base-v2 (768-dim embeddings)


/tmp/ipykernel_4261/4226091034.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Loaded all-mpnet-base-v2 ({embedder.get_sentence_embedding_dimension()}-dim embeddings)")


## 3. Sanity-Check Set: Does Cosine Similarity Actually Track Relevance?

Before trusting embedding similarity as a metric, we validate it against pairs with known, unambiguous relevance labels. A metric that can't separate an obviously-relevant pair from an obviously-irrelevant one has no business scoring real traces. This mirrors the "empirical validation" step the literature review (Section 2.4, Brabant 2026) flags as missing from a lot of RAG metric tooling.

Each pair has a `label` (`"relevant"` or `"irrelevant"`) so we can check the metric separates the two groups with a clear margin, not just a directional trend.

In [3]:
class SanityPair(BaseModel):
    text_a: str
    text_b: str
    label: str  # "relevant" or "irrelevant"
    note: str


sanity_set: list[SanityPair] = [
    SanityPair(
        text_a="What is the capital of France?",
        text_b="Paris is the capital and most populous city of France.",
        label="relevant",
        note="direct factual answer",
    ),
    SanityPair(
        text_a="What is the capital of France?",
        text_b="Bananas are a good source of potassium and dietary fiber.",
        label="irrelevant",
        note="completely unrelated domain",
    ),
    SanityPair(
        text_a="How does photosynthesis work?",
        text_b="Photosynthesis converts light energy into chemical energy stored in glucose.",
        label="relevant",
        note="direct definitional answer",
    ),
    SanityPair(
        text_a="How does photosynthesis work?",
        text_b="The stock market closed lower today amid inflation concerns.",
        label="irrelevant",
        note="completely unrelated domain",
    ),
    SanityPair(
        text_a="NiriZan tracks async context propagation using contextvars.",
        text_b="Based on the context, NiriZan tracks async context for query 'x'.",
        label="relevant",
        note="paraphrase / near-duplicate, should score very high",
    ),
    SanityPair(
        text_a="NiriZan tracks async context propagation using contextvars.",
        text_b="The recipe calls for two cups of flour and a teaspoon of salt.",
        label="irrelevant",
        note="completely unrelated domain",
    ),
]


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity, clipped to [0, 1] since embeddings can yield small
    negative values that don't have a meaningful interpretation as a
    relevance score (see contracts.md: MetricResult.score must be in [0,1])."""
    sim = float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    return max(0.0, min(1.0, (sim + 1) / 2))  # map [-1,1] -> [0,1]


results = []
for pair in sanity_set:
    emb_a, emb_b = embedder.encode([pair.text_a, pair.text_b])
    score = cosine_similarity(emb_a, emb_b)
    results.append((pair.label, score, pair.note))
    print(f"[{pair.label:11}] score={score:.4f}  ({pair.note})")

relevant_scores = [s for label, s, _ in results if label == "relevant"]
irrelevant_scores = [s for label, s, _ in results if label == "irrelevant"]

print(f"\nRelevant pairs   — mean: {np.mean(relevant_scores):.4f}, min: {min(relevant_scores):.4f}")
print(
    f"Irrelevant pairs — mean: {np.mean(irrelevant_scores):.4f}, max: {max(irrelevant_scores):.4f}"
)

margin = min(relevant_scores) - max(irrelevant_scores)
print(f"\nSeparation margin (min relevant - max irrelevant): {margin:.4f}")
assert margin > 0.1, (
    f"Sanity check FAILED: embedding similarity does not cleanly separate relevant from irrelevant pairs (margin={margin:.4f})"
)
print(
    "✅ Sanity check passed: embedding similarity cleanly separates relevant from irrelevant pairs."
)

[relevant   ] score=0.8550  (direct factual answer)
[irrelevant ] score=0.5061  (completely unrelated domain)
[relevant   ] score=0.8406  (direct definitional answer)
[irrelevant ] score=0.4529  (completely unrelated domain)
[relevant   ] score=0.8465  (paraphrase / near-duplicate, should score very high)
[irrelevant ] score=0.4688  (completely unrelated domain)

Relevant pairs   — mean: 0.8474, min: 0.8406
Irrelevant pairs — mean: 0.4759, max: 0.5061

Separation margin (min relevant - max irrelevant): 0.3345
✅ Sanity check passed: embedding similarity cleanly separates relevant from irrelevant pairs.


## 4. The `Metric` Plugin Interface

Per `docs/contracts.md` (Phase 2 Contracts), `Metric` is the interface every future metric implements unchanged, from the RAG Triad here through Phase 4's judges and Phase 5's `BehavioralAnchorMetric`. Two guarantees matter most:
- `score` is always normalized to `[0.0, 1.0]`, regardless of the metric's native scoring range.
- A `Metric` never talks to `regression/`, `gate/`, or `reporting/` — it returns `MetricResult` objects and stops. Persistence is the Metric Dispatcher's job, not the metric's.

In [4]:
class MetricResult(BaseModel):
    model_config = ConfigDict(strict=True)

    metric_name: str
    trace_id: UUID
    score: float = Field(ge=0.0, le=1.0)
    confidence: float | None = Field(default=None, ge=0.0, le=1.0)
    details: dict[str, str | int | float | bool] = Field(default_factory=dict)
    computed_at: datetime


class Metric(Protocol):
    name: str

    async def evaluate(self, trace: Trace) -> list[MetricResult]:
        """Compute one or more scores for a trace.

        Must not mutate the trace. Must not perform its own persistence;
        the Metric Dispatcher is responsible for writing results to
        storage, never the metric itself.
        """
        ...


print("Metric protocol and MetricResult defined.")

Metric protocol and MetricResult defined.


## 5. RAG Triad Metric

Three reference-free scores, computed per trace:

- **Context Relevance**: how relevant the retrieved documents (`RETRIEVAL` span's `output_payload`) are to the original query (root `PLANNING` span's `input_payload`).
- **Groundedness**: how well the generated answer (`GENERATION` span's `output_payload`) is supported by the retrieved context. This is the faithfulness check — an answer can be relevant to the query but still hallucinate content not present in the retrieved docs.
- **Answer Relevance**: how relevant the generated answer is to the original query, independent of whether it's grounded.

Each score is embedding cosine similarity between the relevant text pair, using the sanity-checked scorer from Section 3. All three come back as **separate `MetricResult` objects** with the same `trace_id`, per `docs/contracts.md`'s note that "a single metric module may return multiple `MetricResult`s."

In [5]:
class RAGTriadMetric:
    """Reference-free RAG Triad: context relevance, groundedness, answer relevance.

    Implements the Metric protocol. Scoring backend is pluggable via the
    `scorer` callable (text, text) -> float in [0, 1], so the same class
    works with the embedding scorer here or a Gemini-judge scorer later
    (Section 7) without changing this class.
    """

    name = "rag_triad"

    def __init__(self, scorer):
        self._scorer = scorer

    def _extract_rag_fields(self, trace: Trace) -> dict[str, str | None]:
        """Pull query, context, and answer text out of a trace's spans.

        Query comes from the root PLANNING span's input_payload (the
        original user query, before any pipeline processing). Context
        comes from the RETRIEVAL span's output_payload (the retrieved
        docs). Answer comes from the GENERATION span's output_payload.
        Returns None for any field that isn't present, so evaluate() can
        decide how to handle a trace missing one of the three pieces
        rather than silently scoring against empty strings.
        """
        planning_spans = trace.spans_of_kind(SpanKind.PLANNING)
        retrieval_spans = trace.spans_of_kind(SpanKind.RETRIEVAL)
        generation_spans = trace.spans_of_kind(SpanKind.GENERATION)

        query = planning_spans[0].input_payload if planning_spans else None
        context = retrieval_spans[0].output_payload if retrieval_spans else None
        answer = generation_spans[0].output_payload if generation_spans else None

        return {"query": query, "context": context, "answer": answer}

    async def evaluate(self, trace: Trace) -> list[MetricResult]:
        fields = self._extract_rag_fields(trace)
        query, context, answer = fields["query"], fields["context"], fields["answer"]
        now = datetime.now(timezone.utc)
        results: list[MetricResult] = []

        missing = [k for k, v in fields.items() if v is None]
        if missing:
            # A trace missing a required field can't be scored honestly.
            # Rather than guess with an empty string (which would silently
            # produce a low, misleading score), we skip that specific
            # score and record why in `details`.
            details = {"missing_fields": ",".join(missing)}
        else:
            details = {}

        if query is not None and context is not None:
            score = self._scorer(query, context)
            results.append(
                MetricResult(
                    metric_name="context_relevance",
                    trace_id=trace.trace_id,
                    score=score,
                    computed_at=now,
                    details=details,
                )
            )

        if context is not None and answer is not None:
            score = self._scorer(context, answer)
            results.append(
                MetricResult(
                    metric_name="groundedness",
                    trace_id=trace.trace_id,
                    score=score,
                    computed_at=now,
                    details=details,
                )
            )

        if query is not None and answer is not None:
            score = self._scorer(query, answer)
            results.append(
                MetricResult(
                    metric_name="answer_relevance",
                    trace_id=trace.trace_id,
                    score=score,
                    computed_at=now,
                    details=details,
                )
            )

        return results


def embedding_scorer(text_a: str, text_b: str) -> float:
    emb_a, emb_b = embedder.encode([text_a, text_b])
    return cosine_similarity(emb_a, emb_b)


rag_triad = RAGTriadMetric(scorer=embedding_scorer)
print(f"RAGTriadMetric instantiated with embedding_scorer. name={rag_triad.name!r}")

RAGTriadMetric instantiated with embedding_scorer. name='rag_triad'


## 6. Generate Real Traces from `rag_pipeline_demo` and Score Them

Run the actual instrumented pipeline from `examples/rag_pipeline_demo/main.py`, persist traces through the real `SQLiteTraceRepository`, then run `RAGTriadMetric` against them. This is the first point where Phase 2 metrics touch real Phase 1 infrastructure end-to-end, not synthetic data.

In [6]:
@trace_span(kind=SpanKind.RETRIEVAL, name="qdrant_vector_search")
async def retrieve_context(query: str) -> list[str]:
    await asyncio.sleep(0.02)
    return [
        "NiriZan provides thread/async-safe execution context tracing using contextvars.",
        "Continuous evaluation engines rely on non-blocking trace collectors to avoid adding latency.",
    ]


@trace_span(kind=SpanKind.GENERATION, name="openai_gpt4_inference")
async def generate_response(query: str, documents: list[str]) -> str:
    await asyncio.sleep(0.02)
    return "NiriZan uses Python's contextvars module to propagate trace and span context safely across async calls without blocking the application."


@trace_span(kind=SpanKind.PLANNING, name="rag_orchestrator")
async def run_rag_pipeline(user_query: str) -> str:
    docs = await retrieve_context(user_query)
    return await generate_response(user_query, docs)


async def generate_and_score():
    repo = SQLiteTraceRepository(db_path=":memory:")
    collector = TraceCollector(repository=repo)
    await collector.start()

    exporter = CollectorExporter(collector)
    init_tracer(application_name="rag_pipeline_demo", exporter=exporter)

    queries = [
        "How does NiriZan handle context propagation?",
        "What is the role of the TraceCollector?",
    ]
    for q in queries:
        await run_rag_pipeline(q)

    await collector.stop()

    traces = await repo.list_by_application("rag_pipeline_demo")
    print(f"Generated and persisted {len(traces)} real traces.\n")

    all_results: list[MetricResult] = []
    for trace in traces:
        metric_results = await rag_triad.evaluate(trace)
        all_results.extend(metric_results)
        print(f"Trace {str(trace.trace_id)[:8]}...")
        for r in metric_results:
            print(f"   {r.metric_name:18} score={r.score:.4f}")
        print()

    return traces, all_results


demo_traces, demo_results = await generate_and_score()

Generated and persisted 2 real traces.

Trace eeade586...
   context_relevance  score=0.6314
   groundedness       score=0.8317
   answer_relevance   score=0.6233

Trace 34397429...
   context_relevance  score=0.6949
   groundedness       score=0.8317
   answer_relevance   score=0.8208



## 7. Optional: Gemini LLM-as-Judge Cross-Check

The embedding scorer above is the default, free, reference-free scoring backend. This section adds an **optional** Gemini-based judge to cross-check a subset of scores, closer to how RAGAS/ARES actually compute the RAG Triad in practice (prompted judgment, not just cosine similarity).

Requires a free Gemini API key from [Google AI Studio](https://aistudio.google.com/apikey). If you don't have one, skip this section — the embedding scorer above is fully functional on its own and is what the Phase 2 `metrics/rag_triad.py` will ship with by default.

In [7]:
import os
from google import genai

GEMINI_API_KEY = (
    os.environ.get("GEMINI_API_KEY") or ""
)  # or: input("Gemini API key (blank to skip): ")

gemini_client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None

JUDGE_PROMPT_TEMPLATE = """You are scoring semantic relevance between two texts on a scale of 0.0 to 1.0.
0.0 means completely unrelated. 1.0 means text B fully and directly addresses/supports text A.
Respond with ONLY a single float between 0.0 and 1.0, nothing else.

Text A: {text_a}
Text B: {text_b}

Score:"""


def gemini_scorer(text_a: str, text_b: str) -> float:
    if gemini_client is None:
        raise RuntimeError("Gemini client not configured; set GEMINI_API_KEY to use this scorer.")
    response = gemini_client.models.generate_content(
        model="gemini-2.0-flash",
        contents=JUDGE_PROMPT_TEMPLATE.format(text_a=text_a, text_b=text_b),
    )
    try:
        score = float(response.text.strip())
        return max(0.0, min(1.0, score))
    except (ValueError, AttributeError):
        raise ValueError(f"Gemini returned a non-numeric response: {response.text!r}")


if gemini_client:
    gemini_rag_triad = RAGTriadMetric(scorer=gemini_scorer)
    trace_to_check = demo_traces[0]
    gemini_results = await gemini_rag_triad.evaluate(trace_to_check)

    print(f"Cross-check for trace {str(trace_to_check.trace_id)[:8]}...\n")
    print(f"{'metric':18} {'embedding':>10} {'gemini':>10} {'diff':>8}")
    embedding_by_name = {
        r.metric_name: r.score for r in demo_results if r.trace_id == trace_to_check.trace_id
    }
    for r in gemini_results:
        emb_score = embedding_by_name.get(r.metric_name, float("nan"))
        print(
            f"{r.metric_name:18} {emb_score:>10.4f} {r.score:>10.4f} {abs(emb_score - r.score):>8.4f}"
        )
else:
    print(
        "No GEMINI_API_KEY set — skipping optional Gemini cross-check. Embedding scorer results from Section 6 stand on their own."
    )

No GEMINI_API_KEY set — skipping optional Gemini cross-check. Embedding scorer results from Section 6 stand on their own.


## 8. Metric Dispatcher

Routes traces to the correct metric(s) based on system type. Registration is explicit (per `docs/contracts.md`: "there is no implicit auto-discovery of metrics on import"), so `metrics/` never needs to be imported wholesale by `orchestrator/`.

In [8]:
class MetricDispatcher:
    def __init__(self):
        self._registry: dict[str, list[Metric]] = {}

    def register(self, metric: Metric, applies_to: set[str]) -> None:
        for system_type in applies_to:
            self._registry.setdefault(system_type, []).append(metric)

    async def dispatch(self, trace: Trace, system_type: str) -> list[MetricResult]:
        metrics = self._registry.get(system_type, [])
        all_results: list[MetricResult] = []
        for metric in metrics:
            results = await metric.evaluate(trace)
            all_results.extend(results)
        return all_results


dispatcher = MetricDispatcher()
dispatcher.register(rag_triad, applies_to={"rag_pipeline"})

print(
    f"Registered metrics for 'rag_pipeline': {[m.name for m in dispatcher._registry['rag_pipeline']]}"
)

dispatch_results = await dispatcher.dispatch(demo_traces[0], system_type="rag_pipeline")
print(
    f"\nDispatched {len(dispatch_results)} MetricResults for trace {str(demo_traces[0].trace_id)[:8]}..."
)
for r in dispatch_results:
    print(f"   {r.metric_name:18} score={r.score:.4f}")

Registered metrics for 'rag_pipeline': ['rag_triad']

Dispatched 3 MetricResults for trace eeade586...
   context_relevance  score=0.6314
   groundedness       score=0.8317
   answer_relevance   score=0.6233


## 9. Run Scheduler (On-Demand)

Minimal on-demand scheduler per Phase 2 deliverables. Triggers an evaluation run: pulls traces from the repository, dispatches them through the Metric Dispatcher, and returns the results ready to be persisted against a `Run` (Phase 3 territory — here we just prove the trigger works).

In [9]:
class RunScheduler:
    def __init__(self, repository: SQLiteTraceRepository, dispatcher: MetricDispatcher):
        self.repository = repository
        self.dispatcher = dispatcher

    async def run_on_demand(
        self, application_name: str, system_type: str
    ) -> dict[UUID, list[MetricResult]]:
        traces = await self.repository.list_by_application(application_name)
        results_by_trace: dict[UUID, list[MetricResult]] = {}
        for trace in traces:
            results_by_trace[trace.trace_id] = await self.dispatcher.dispatch(trace, system_type)
        return results_by_trace


async def demo_scheduler():
    repo = SQLiteTraceRepository(db_path=":memory:")
    collector = TraceCollector(repository=repo)
    await collector.start()
    exporter = CollectorExporter(collector)
    init_tracer(application_name="rag_pipeline_demo", exporter=exporter)

    await run_rag_pipeline("How does the scheduler trigger an evaluation run?")
    await collector.stop()

    scheduler = RunScheduler(repository=repo, dispatcher=dispatcher)
    run_results = await scheduler.run_on_demand(
        application_name="rag_pipeline_demo", system_type="rag_pipeline"
    )

    for trace_id, results in run_results.items():
        print(f"Run triggered on-demand for trace {str(trace_id)[:8]}...")
        for r in results:
            print(f"   {r.metric_name:18} score={r.score:.4f}")

    return run_results


scheduler_results = await demo_scheduler()
print("\n✅ RunScheduler.run_on_demand() verified end-to-end: trace → dispatch → MetricResults.")

Run triggered on-demand for trace 17f3a3a3...
   context_relevance  score=0.6216
   groundedness       score=0.8317
   answer_relevance   score=0.6044

✅ RunScheduler.run_on_demand() verified end-to-end: trace → dispatch → MetricResults.
